JOB ACCEPTANCE PREDICTION - FEATURE ENGINEERING

In [52]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [53]:
import sys
print("Python Path:", sys.executable)
print("Version:", sys.version)


Python Path: c:\Users\user\AppData\Local\Programs\Python\Python311\python.exe
Version: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]


In [54]:
import sys
!{sys.executable} -m pip install -U scikit-learn


What i did : Loaded the cleaned HR dataset and verified environment & library versions

In [55]:
df = pd.read_csv("../data/processed/cleaned_hr_data.csv")
print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Data loaded: 50,124 rows, 26 columns


In [56]:
df['academic_index'] = (
    df['ssc_percentage'] * 0.25 +
    df['hsc_percentage'] * 0.35 +
    df['degree_percentage'] * 0.40
)


In [57]:
df['overall_assessment'] = (
    df['technical_score'] * 0.40 +
    df['aptitude_score'] * 0.30 +
    df['communication_score'] * 0.30
)


In [58]:
df['ctc_gap'] = df['expected_ctc_lpa'] - df['previous_ctc_lpa']
df['ctc_gap_percentage'] = (
    df['ctc_gap'] / df['previous_ctc_lpa'].replace(0, 1)
) * 100


In [59]:
df['experience_category'] = pd.cut(
    df['years_of_experience'],
    bins=[-1, 0, 2, 5, 10, 100],
    labels=['Fresher', 'Junior', 'Mid', 'Senior', 'Expert']
)


In [60]:
df['skills_match_level'] = pd.cut(
    df['skills_match_percentage'],
    bins=[0, 50, 75, 90, 100],
    labels=['Low', 'Medium', 'High', 'Excellent']
)


In [61]:
df['interview_score'] = (
    df['technical_score'] * 0.5 +
    df['communication_score'] * 0.3 +
    df['aptitude_score'] * 0.2
)


In [62]:
df['total_score'] = (
    df['academic_index'] * 0.2 +
    df['overall_assessment'] * 0.4 +
    df['skills_match_percentage'] * 0.3 +
    df['certifications_count'] * 0.1
)


In [63]:
df['risk_score'] = (
    (df['layoff_history'] == 'Yes').astype(int) * 0.3 +
    (df['employment_gap_months'] > 12).astype(int) * 0.3 +
    (df['notice_period_days'] > 60).astype(int) * 0.2 +
    (df['ctc_gap_percentage'] > 50).astype(int) * 0.2
)


In [64]:
df['skills_tech_ratio'] = df['skills_match_percentage'] / df['technical_score'].replace(0, 1)
df['exp_salary_ratio'] = df['years_of_experience'] / df['expected_ctc_lpa'].replace(0, 1)


In [65]:
print(f" New dataset shape: {df.shape}")


 New dataset shape: (50124, 37)


In [66]:
binary_features = [
    'gender', 'internship_experience', 'career_switch_willingness',
    'relevant_experience', 'job_role_match', 'bond_requirement',
    'layoff_history', 'relocation_willingness'
]

le = LabelEncoder()
for col in binary_features:
    if col in df.columns:
        df[col] = le.fit_transform(df[col])


In [67]:
multi_cat_features = [ 'degree_specialization', 'company_tier', 'competition_level', 'experience_category', 'skills_match_level' ] 
df = pd.get_dummies(df, columns=multi_cat_features, drop_first=True) 
print("✓ Categorical encoding completed")



✓ Categorical encoding completed


In [68]:
df['target'] = df['status'].map({'Placed': 1, 'Not Placed': 0})
print(df['target'].value_counts())


target
0    34950
1    15174
Name: count, dtype: int64


In [69]:
scaler = StandardScaler()

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = numeric_cols.drop(['target'])  # prevent leakage

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print(f" Scaled {len(numeric_cols)} numerical features")


 Scaled 31 numerical features


In [70]:
df['target'].value_counts()


target
0    34950
1    15174
Name: count, dtype: int64

In [71]:
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Correlation with target
corr = numeric_df.corr()['target'].abs().sort_values(ascending=False)

print("\nTop 15 Features correlated with placement:")
for i, col in enumerate(corr.index[1:16], 1):  # skip target itself
    actual_corr = numeric_df[['target', col]].corr().iloc[0, 1]
    sign = "+" if actual_corr > 0 else "-"
    print(f"{i:2}. {col:30} : {sign}{abs(actual_corr):.3f}")



Top 15 Features correlated with placement:
 1. technical_score                : +0.404
 2. interview_score                : +0.399
 3. overall_assessment             : +0.367
 4. total_score                    : +0.365
 5. years_of_experience            : +0.331
 6. exp_salary_ratio               : +0.328
 7. skills_match_percentage        : +0.198
 8. risk_score                     : -0.196
 9. ctc_gap                        : -0.190
10. job_role_match                 : -0.187
11. skills_tech_ratio              : -0.185
12. expected_ctc_lpa               : -0.162
13. ctc_gap_percentage             : -0.149
14. relevant_experience            : +0.138
15. internship_experience          : +0.128


In [72]:
df.columns

Index(['age_years', 'gender', 'ssc_percentage', 'hsc_percentage',
       'degree_percentage', 'technical_score', 'aptitude_score',
       'communication_score', 'skills_match_percentage',
       'certifications_count', 'internship_experience', 'years_of_experience',
       'career_switch_willingness', 'relevant_experience', 'previous_ctc_lpa',
       'expected_ctc_lpa', 'job_role_match', 'bond_requirement',
       'notice_period_days', 'layoff_history', 'employment_gap_months',
       'relocation_willingness', 'status', 'academic_index',
       'overall_assessment', 'ctc_gap', 'ctc_gap_percentage',
       'interview_score', 'total_score', 'risk_score', 'skills_tech_ratio',
       'exp_salary_ratio', 'degree_specialization_Electronics',
       'degree_specialization_Information Technology',
       'degree_specialization_Mechanical', 'degree_specialization_Others',
       'company_tier_Tier 1 ', 'company_tier_Tier 2', 'company_tier_Tier 2 ',
       'company_tier_Tier 3', 'company_tier_Ti

In [73]:
output_path = "../data/processed/hr_data_engineered.csv"
df.to_csv(output_path, index=False)

print(" Feature engineering completed")
print("Final shape:", df.shape)


 Feature engineering completed
Final shape: (50124, 51)


In [74]:
df.dtypes.value_counts()


float64    31
bool       18
object      1
int64       1
Name: count, dtype: int64

In [75]:
df['target'].value_counts()

target
0    34950
1    15174
Name: count, dtype: int64